<div style="background:#0A005A;border-radius:18px;padding:28px 32px;color:#fff;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif">

<div style="margin-top:22px"><span style="background:#613EFF;color:#fff;padding:5px 12px;border-radius:999px;font-size:.8em;font-weight:700;letter-spacing:.06em">TUTORIAL · NOTEBOOK</span>
<span style="color:#C9C2FF;font-size:.85em;margin-left:8px">Engenharia de Dados · Nível 0 · Fundamentos</span></div>
<h1 style="color:#fff;margin:14px 0 8px;font-size:2.1em;line-height:1.15">Prática: testar o dado</h1>
<p style="color:#E4E0FF;margin:0 0 12px;font-size:1.05em">Escreva as consultas que procuram problemas e veja cada uma pegar um defeito de verdade.</p>
<span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">cerca de 25 minutos</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">Python 3.10+</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">roda no Colab</span><span style="display:inline-block;background:rgba(255,255,255,.12);color:#fff;padding:5px 12px;border-radius:999px;margin:4px 6px 0 0;font-size:.85em">dados fictícios</span>
</div>

## O que você vai fazer

- Escrever testes como consultas que procuram linhas com problema.
- Provar que um teste funciona, fazendo-o falhar de propósito.
- Separar o que é erro (para tudo) do que é aviso (alguém precisa olhar).
- Conferir que a gold não perdeu nem ganhou viagens: a reconciliação.

A explicação completa está na aula: [Prática: testar o dado](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-testes.md).

## Como usar este notebook

Rode as células **em ordem**, de cima para baixo: clique na célula e aperte <kbd>Shift</kbd> + <kbd>Enter</kbd>. A primeira linha de cada célula diz o que fazer com ela:

| Começa com | O que fazer |
|---|---|
| `# Rode e observe` | Código pronto. Rode, leia a saída e compare com a explicação acima dela. |
| `# Sua vez` | Complete o código onde está `None` e rode. |
| `# Confira` | Rode depois da sua vez: ela mostra `[OK]` se está certo, `[ERRO]` com o que falta, ou `[AVISO]` se você ainda não respondeu. |

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Onde rodar</b><br>No <b>Google Colab</b> (só abrir, sem instalar nada), no <b>VS Code</b> com a extensão Jupyter ou no <b>JupyterLab</b> (<code>pip install jupyterlab</code> e depois <code>jupyter lab</code>). Os dados são fictícios e ficam no próprio notebook: nada vai para fora da sua máquina.</div>

##  Preparação

Instale os pacotes, carregue os quadros de conferência e crie o cenário da Pedala Aurora.

In [ ]:
%pip install -q duckdb httpx pandas

In [ ]:
from IPython.display import HTML, display


def mostrar(tipo, titulo, detalhe=""):
    """Mostra um quadro colorido com o resultado de uma conferência."""
    cor, rotulo = {"ok": ("#613EFF", "[OK]"), "erro": ("#F9006F", "[ERRO]"), "info": ("#0A005A", "[AVISO]")}[tipo]
    display(HTML(
        f'<div style="border-left:5px solid {cor};background:#F2F1F6;padding:10px 14px;'
        f'border-radius:8px;margin:6px 0;color:#202020"><b style="color:{cor}">{rotulo} {titulo}</b>'
        + (f"<br>{detalhe}" if detalhe else "") + "</div>"))


def conferir(resposta, certo, mensagem_certo, mensagem_errado):
    """Compara a sua resposta com a esperada e mostra um quadro verde, rosa ou de aviso."""
    if resposta is None:
        mostrar("info", "Ainda sem resposta", "Complete a célula “Sua vez” acima, rode-a e rode esta de novo.")
    elif certo:
        mostrar("ok", mensagem_certo)
    else:
        mostrar("erro", mensagem_errado)


mostrar("ok", "Quadros de conferência prontos")

A célula abaixo gera os dados e liga uma **API de mentira** aqui mesmo, no notebook. Ela está recolhida porque você não precisa ler o código para seguir; se tiver curiosidade, abra.

In [ ]:
#@title Preparar o cenário (clique duas vezes para ver o código)
# Cria os dados da Pedala Aurora e liga, aqui mesmo no notebook, uma API de mentira que os entrega.
# Os dados são sintéticos e sempre os mesmos (semente fixa). Você não precisa ler este código para seguir.
import json
import random
import threading
from datetime import date, datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

FUSO = timezone(timedelta(hours=-3))
ESTACOES = [
    {"id_estacao": "E01", "nome": "Praça Central", "bairro": "Centro", "vagas": 20},
    {"id_estacao": "E02", "nome": "Rodoviária", "bairro": "Centro", "vagas": 15},
    {"id_estacao": "E03", "nome": "Parque das Águas", "bairro": "Jardim Norte", "vagas": 12},
    {"id_estacao": "E04", "nome": "Mercado Municipal", "bairro": "Centro", "vagas": 16},
    {"id_estacao": "E05", "nome": "Universidade", "bairro": "Vila Nova", "vagas": 18},
    {"id_estacao": "E06", "nome": "Hospital Regional", "bairro": "Vila Nova", "vagas": 10},
    {"id_estacao": "E07", "nome": "Orla do Lago", "bairro": "Lagoa", "vagas": 14},
    {"id_estacao": "E08", "nome": "Estação do Metrô", "bairro": "Jardim Sul", "vagas": 20},
]
PRIMEIRO_DIA, ULTIMO_DIA = date(2026, 6, 1), date(2026, 6, 7)
POR_PAGINA = 40


def _gerar():
    sorteio = random.Random(2026)
    pesos = [5, 3, 2, 3, 4, 1, 2, 5]
    horas = list(range(6, 23))
    peso_hora = [2, 6, 9, 5, 3, 3, 4, 3, 3, 3, 4, 7, 9, 6, 3, 2, 1]
    viagens, atrasadas, numero = {}, [], 5000
    dia = PRIMEIRO_DIA
    while dia <= ULTIMO_DIA:
        quantas = sorteio.randint(85, 115) if dia.weekday() < 5 else sorteio.randint(55, 75)
        lista = []
        for _ in range(quantas):
            numero += 1
            saida, chegada = sorteio.choices(ESTACOES, pesos, k=2)
            hora = sorteio.choices(horas, peso_hora)[0]
            inicio = datetime(dia.year, dia.month, dia.day, hora, sorteio.randint(0, 59), tzinfo=FUSO)
            fim = inicio + timedelta(minutes=sorteio.randint(3, 45))
            lista.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": saida["id_estacao"], "estacao_chegada": chegada["id_estacao"],
                          "inicio": inicio.isoformat(), "fim": fim.isoformat()})
        lista.sort(key=lambda v: v["inicio"])
        viagens[dia.isoformat()] = lista
        dia += timedelta(days=1)
    # Defeitos plantados, como os de uma fonte real:
    for v in viagens["2026-06-04"][10:12]:                 # fim antes do início
        v["inicio"], v["fim"] = v["fim"], v["inicio"]
    for dia_, i in (("2026-06-02", 30), ("2026-06-04", 50), ("2026-06-06", 20)):
        viagens[dia_][i]["estacao_saida"] = "E99"           # estação fora do cadastro
    for _ in range(4):                                      # viagens de 05/06 que só chegam depois
        numero += 1
        inicio = datetime(2026, 6, 5, 21, sorteio.randint(0, 59), tzinfo=FUSO)
        atrasadas.append({"id_viagem": f"V{numero}", "id_usuario": f"U{sorteio.randint(100, 399)}",
                          "estacao_saida": "E07", "estacao_chegada": "E01",
                          "inicio": inicio.isoformat(), "fim": (inicio + timedelta(minutes=12)).isoformat()})
    return viagens, atrasadas


VIAGENS, _ATRASADAS = _gerar()
_estado = {"atrasadas_chegaram": False, "falha_temporaria": True}


def chegam_viagens_atrasadas():
    """Simula o dia seguinte: a fonte recebe 4 viagens de 05/06 que ainda não tinham sido enviadas."""
    if not _estado["atrasadas_chegaram"]:
        VIAGENS["2026-06-05"] = sorted(VIAGENS["2026-06-05"] + _ATRASADAS, key=lambda v: v["inicio"])
        _estado["atrasadas_chegaram"] = True


class _Api(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def _responder(self, status, corpo=None):
        self.send_response(status)
        if corpo is None:
            self.end_headers()
            return
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        args = {k: v[0] for k, v in parse_qs(url.query).items()}
        if url.path == "/estacoes":
            return self._responder(200, ESTACOES)
        if url.path != "/viagens":
            return self._responder(404, {"erro": "caminho inexistente"})
        try:
            dia = date.fromisoformat(args.get("data", ""))
            pagina = int(args.get("pagina", "1"))
        except ValueError:
            return self._responder(400, {"erro": "use data=AAAA-MM-DD e pagina=1, 2, ..."})
        lista = VIAGENS.get(dia.isoformat(), [])
        if not lista:
            return self._responder(204)
        if dia.isoformat() == "2026-06-02" and pagina == 2 and _estado["falha_temporaria"]:
            _estado["falha_temporaria"] = False
            return self._responder(503, {"erro": "servidor ocupado, tente de novo"})
        paginas = -(-len(lista) // POR_PAGINA)
        inicio = (pagina - 1) * POR_PAGINA
        trecho = lista[inicio:inicio + POR_PAGINA]
        if dia.isoformat() == "2026-06-03" and pagina == 2:
            trecho = [lista[inicio - 1]] + trecho               # a página 2 repete o último da página 1
        self._responder(200, {"data": dia.isoformat(), "pagina": pagina, "total_paginas": paginas,
                              "total_registros": len(lista), "viagens": trecho})


_servidor = ThreadingHTTPServer(("127.0.0.1", 0), _Api)
threading.Thread(target=_servidor.serve_forever, daemon=True).start()
API = f"http://127.0.0.1:{_servidor.server_port}"
print(f"Cenário pronto: {len(ESTACOES)} estações, {sum(len(v) for v in VIAGENS.values())} viagens "
      f"de {PRIMEIRO_DIA:%d/%m} a {ULTIMO_DIA:%d/%m}. API em {API}")

Esta célula refaz a coleta, a gravação e as camadas das aulas anteriores, num banco na memória.

In [ ]:
import time

import httpx


def pedir(caminho, parametros=None, tentativas=3):
    """Faz um pedido à API. Tenta de novo quando o erro é do servidor (5xx)."""
    for tentativa in range(1, tentativas + 1):
        resposta = httpx.get(API + caminho, params=parametros, timeout=10)
        if resposta.status_code == 204:
            return None                      # dia sem viagens: a resposta vem vazia
        if resposta.status_code >= 500 and tentativa < tentativas:
            print(f"  {resposta.status_code} na tentativa {tentativa}: nova tentativa em 1 segundo")
            time.sleep(1)
            continue
        resposta.raise_for_status()          # 4xx: o pedido está errado; insistir não resolve
        return resposta.json()


def coletar_dia(data):
    """Todas as viagens de um dia, página por página, e o total que a API informa."""
    viagens, pagina, total_paginas, total_registros = [], 1, 1, 0
    while pagina <= total_paginas:
        corpo = pedir("/viagens", {"data": data, "pagina": pagina})
        if corpo is None:                    # 204: não há viagens neste dia
            return [], 0
        viagens.extend(corpo["viagens"])
        total_paginas, total_registros = corpo["total_paginas"], corpo["total_registros"]
        pagina += 1
    return viagens, total_registros


import json
from datetime import datetime
from pathlib import Path

PASTA = Path("bruto")


def salvar_bruto(data, viagens, total_registros):
    """Guarda o dia exatamente como veio da API, com a hora da coleta."""
    PASTA.mkdir(exist_ok=True)
    arquivo = PASTA / f"viagens_{data}.json"
    conteudo = {"data": data, "coletado_em": datetime.now().isoformat(timespec="seconds"),
                "total_registros": total_registros, "viagens": viagens}
    arquivo.write_text(json.dumps(conteudo, ensure_ascii=False), encoding="utf-8")
    return arquivo


DIAS = [f"2026-06-0{d}" for d in range(1, 8)]   # de 01/06 a 07/06


for data in DIAS:
    salvar_bruto(data, *coletar_dia(data))
print(f"Pasta bruto/ pronta: {len(list(PASTA.glob('*.json')))} arquivos")


import duckdb

con = duckdb.connect()                       # banco na memória: some quando o notebook fecha
con.execute("set TimeZone = 'America/Sao_Paulo'")   # datas e horas mostradas no horário de Brasília


con.execute("""
    create table if not exists raw_viagens (
        data_ref        varchar,
        id_viagem       varchar,
        id_usuario      varchar,
        estacao_saida   varchar,
        estacao_chegada varchar,
        inicio          varchar,
        fim             varchar,
        coletado_em     varchar
    )
""")


def linhas_do_arquivo(arquivo):
    """Lê um arquivo da pasta bruto/ e devolve as linhas da tabela raw_viagens."""
    conteudo = json.loads(arquivo.read_text(encoding="utf-8"))
    return conteudo["data"], [
        (conteudo["data"], v["id_viagem"], v["id_usuario"], v["estacao_saida"],
         v["estacao_chegada"], v["inicio"], v["fim"], conteudo["coletado_em"])
        for v in conteudo["viagens"]
    ]


def gravar_dia(con, arquivo):
    """Substitui o dia inteiro: apaga o que havia daquele dia e grava de novo, numa transação."""
    data, linhas = linhas_do_arquivo(arquivo)
    con.execute("begin")
    try:
        con.execute("delete from raw_viagens where data_ref = ?", [data])
        con.executemany("insert into raw_viagens values (?, ?, ?, ?, ?, ?, ?, ?)", linhas)
        con.execute("commit")
    except Exception:
        con.execute("rollback")              # desfaz o delete: o dia antigo volta inteiro
        raise
    return len(linhas)


def gravar_estacoes(con):
    """O cadastro é pequeno: a cada carga, substitui a tabela inteira."""
    estacoes = pedir("/estacoes")
    con.execute("create or replace table raw_estacoes (id_estacao varchar, nome varchar, bairro varchar, vagas varchar)")
    con.executemany("insert into raw_estacoes values (?, ?, ?, ?)",
                    [(e["id_estacao"], e["nome"], e["bairro"], str(e["vagas"])) for e in estacoes])
    return len(estacoes)


for arquivo in sorted(PASTA.glob("*.json")):
    gravar_dia(con, arquivo)
gravar_estacoes(con)
print("raw_viagens:", con.execute("select count(*) from raw_viagens").fetchone()[0], "linhas")


con.execute("""create or replace view bronze_viagens as
select
    cast(data_ref as date)        as data_ref,
    id_viagem,
    id_usuario,
    estacao_saida,
    estacao_chegada,
    cast(inicio as timestamptz)   as inicio,
    cast(fim as timestamptz)      as fim,
    cast(coletado_em as timestamp) as coletado_em
from raw_viagens;

create or replace view bronze_estacoes as
select id_estacao, nome, bairro, cast(vagas as integer) as vagas
from raw_estacoes;""")
con.execute("""create or replace view silver_viagens as
with sem_repetidas as (
    select *,
           row_number() over (partition by id_viagem order by coletado_em desc) as ordem
    from bronze_viagens
)
select
    v.id_viagem,
    v.id_usuario,
    v.estacao_saida,
    e.nome                                  as estacao_saida_nome,
    e.bairro                                as estacao_saida_bairro,
    v.estacao_chegada,
    v.inicio,
    v.fim,
    datediff('minute', v.inicio, v.fim)     as duracao_min,
    e.id_estacao is null                    as estacao_desconhecida
from sem_repetidas v
left join bronze_estacoes e on e.id_estacao = v.estacao_saida
where v.ordem = 1              -- regra 1: uma linha por viagem
  and v.fim >= v.inicio;       -- regra 2: fim antes do início é erro da fonte""")
con.execute("""create or replace table gold_saidas_por_estacao_dia as
select
    cast(inicio as date)                              as dia,
    estacao_saida,
    coalesce(estacao_saida_nome, 'Estação desconhecida') as estacao,
    count(*)                                          as saidas,
    median(duracao_min)                               as duracao_mediana_min
from silver_viagens
group by all
order by dia, saidas desc, estacao_saida;""")
print("gold_saidas_por_estacao_dia:", con.execute("select count(*) from gold_saidas_por_estacao_dia").fetchone()[0], "linhas")

##  Passo 1 de 4 · Um teste é uma consulta

Cada teste procura linhas que **não deveriam existir**. Nenhuma linha: passou. Alguma linha: falhou, e o resultado já mostra quais.

In [ ]:
# Rode e observe
TESTES = {
    "uma linha por viagem": ("erro", """
        select id_viagem from silver_viagens group by id_viagem having count(*) > 1"""),
    "início e fim preenchidos": ("erro", """
        select id_viagem from silver_viagens where inicio is null or fim is null"""),
    "duração não negativa": ("erro", """
        select id_viagem from silver_viagens where duracao_min < 0"""),
    "estação de saída existe no cadastro": ("aviso", """
        select id_viagem from silver_viagens where estacao_desconhecida"""),
}


def rodar_testes(con, testes):
    """Cada teste é uma consulta que procura linhas com problema. Nenhuma linha: passou."""
    falhas = 0
    for nome, (gravidade, sql) in testes.items():
        problemas = len(con.execute(sql).fetchall())
        if problemas == 0:
            mostrar("ok", nome)
        elif gravidade == "aviso":
            mostrar("info", f"{nome}: {problemas} linha(s)", "Aviso: o dado segue, mas alguém precisa olhar.")
        else:
            mostrar("erro", f"{nome}: {problemas} linha(s)")
            falhas += 1
    return falhas


falhas = rodar_testes(con, TESTES)

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Erro ou aviso</b><br>Um <b>erro</b> quer dizer que o dado está errado e não deve seguir: viagem duplicada, duração negativa. Um <b>aviso</b> quer dizer que o dado segue, mas alguém precisa agir: as três viagens da estação E99 existem, o que falta é a estação no cadastro.</div>

##  Passo 2 de 4 · Fazer o teste falhar

Um teste que nunca falhou não prova nada. Rode os mesmos testes contra a **bronze**, que ainda não tem as regras.

In [ ]:
# Rode e observe
con.execute("""
    create or replace temp view silver_sem_regras as
    select *,
           datediff('minute', inicio, fim) as duracao_min,
           estacao_saida not in (select id_estacao from bronze_estacoes) as estacao_desconhecida
    from bronze_viagens
""")
testes_na_bronze = {nome: (gravidade, sql.replace("silver_viagens", "silver_sem_regras"))
                    for nome, (gravidade, sql) in TESTES.items()}
falhas = rodar_testes(con, testes_na_bronze)

<div style="border-left:5px solid #613EFF;background:#F2F1F6;padding:12px 16px;border-radius:10px;margin:10px 0;color:#202020;font-family:'Reddit Sans', 'Open Sans', system-ui, sans-serif"><b style="color:#613EFF">Leia assim</b><br>Sem as regras da silver, o teste de linha única pega a viagem repetida e o de duração pega as duas viagens com fim antes do início. Agora você sabe que esses testes funcionam. O aviso da estação aparece nas duas versões, porque a silver mantém essas viagens de propósito.</div>

##  Passo 3 de 4 · A reconciliação

Os testes acima olham linha por linha. Falta um que olhe o **todo**: a soma das saídas na gold tem que ser igual ao número de viagens na silver. Se não for, a gold perdeu ou inventou viagens.

### Sua vez

Escreva a consulta da reconciliação em `reconciliacao`. Como os outros testes, ela deve devolver **uma linha quando há problema** e nenhuma quando está tudo certo. Dica: compare `sum(saidas)` da gold com `count(*)` da silver.

In [ ]:
# Sua vez
reconciliacao = None   # troque pela consulta, entre três aspas: """select ..."""

In [ ]:
# Confira
certo = None
if reconciliacao is not None:
    certo = len(con.execute(reconciliacao).fetchall()) == 0
    con.execute("create or replace temp table gold_quebrada as select * from gold_saidas_por_estacao_dia where estacao_saida <> 'E99'")
    certo = certo and len(con.execute(reconciliacao.replace("gold_saidas_por_estacao_dia", "gold_quebrada")).fetchall()) == 1
conferir(reconciliacao, certo, "A reconciliação passa na gold certa e falha numa gold que perdeu as viagens da E99",
         "Ainda não: a consulta deve devolver zero linhas com a gold certa e uma linha com a gold quebrada")

<details>
<summary><b>Ver uma resposta</b> (tente antes)</summary>

```python
reconciliacao = """
select * from (
    select
        (select sum(saidas) from gold_saidas_por_estacao_dia) as na_gold,
        (select count(*) from silver_viagens)                  as na_silver
) where na_gold <> na_silver
"""
```
</details>

##  Passo 4 de 4 · Todos os testes juntos

Junte a reconciliação aos outros testes. Esta é a bateria que vai rodar a cada carga, na próxima aula.

In [ ]:
# Rode e observe
TESTES["gold reconcilia com a silver"] = ("erro", """
select * from (
    select
        (select sum(saidas) from gold_saidas_por_estacao_dia) as na_gold,
        (select count(*) from silver_viagens)                  as na_silver
) where na_gold <> na_silver
""")
falhas = rodar_testes(con, TESTES)
mostrar("ok" if falhas == 0 else "erro", "Pode publicar" if falhas == 0 else f"{falhas} erro(s): não publique")

## O que você aprendeu

- Um teste de dado é uma consulta que procura linhas com problema.
- Faça o teste falhar de propósito uma vez, para saber que ele funciona.
- Erro para o pipeline; aviso deixa o dado seguir e chama alguém.
- A reconciliação compara totais entre camadas e pega o que os testes de linha não veem.

**Próxima aula:** [Prática: orquestrar o pipeline](https://gov-hub.io/capacitacao/doc.html?trilha=engenharia-de-dados&path=docs/tutoriais/engenharia-fundamentos-orquestrar.md).